In [ ]:
from pathlib import Path
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW, BASE, OUT, PROCESSED = RAW_DIR, BASE_DIR, PROCESSED_DIR, PROCESSED_DIR
print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

In [ ]:
import requests

SOURCES_URL = {
    "esp8266-technical_reference_en.pdf":
        "https://documentation.espressif.com/esp8266-technical_reference_en.pdf",
    "0c-esp-wroom-02_datasheet_en.pdf":
        "https://www.espressif.com/sites/default/files/documentation/0c-esp-wroom-02_datasheet_en.pdf",
}
for name, url in SOURCES_URL.items():
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    (RAW_DIR / name).write_bytes(r.content)
    print(name, len(r.content), "bytes")

In [ ]:
import re, requests
from datetime import datetime, timezone
from bs4 import BeautifulSoup

SDK_URL = "https://docs.espressif.com/projects/esp8266-rtos-sdk/en/latest/get-started/index.html"
SDK_FILE = "esp8266_rtos_sdk_get_started.txt"

r = requests.get(SDK_URL, timeout=60)
r.raise_for_status()
soup = BeautifulSoup(r.text, "html.parser")

main = soup.find(attrs={"role": "main"}) or soup.find("main") or soup.body
for tag in main(["script", "style", "nav", "footer"]):
    tag.decompose()

def tidy(s):
    return re.sub(r"[\ue000-\uf8ff¶]", "", s).strip()

heads = {tidy(h.get_text("")) for h in main.find_all(["h1", "h2", "h3", "h4"])}
heads.discard("")

# put block elements on their own lines; keep inline text (code spans, links) joined
for el in main.find_all(["p", "li", "pre", "h1", "h2", "h3", "h4", "tr", "dt", "dd"]):
    el.insert_before("\n")
    el.insert_after("\n")
for el in main.find_all(["td", "th"]):
    el.insert_after(" | ")

lines = []
for raw_line in main.get_text("").splitlines():
    line = re.sub(r"[ \t]+", " ", tidy(raw_line))
    if line:
        lines.append(f"## {line}" if line in heads else line)

header = [
    "## Document header",
    f"SOURCE_URL: {SDK_URL}",
    f"RETRIEVED_UTC: {datetime.now(timezone.utc).isoformat(timespec='seconds')}",
    f"HTML_CONTAINER: {main.name} role={main.get('role')}",
]
(RAW_DIR / SDK_FILE).write_text("\n".join(header + lines), encoding="utf-8")

assert len(lines) > 20, "Very little text extracted; inspect the page structure."
print("Lines:", len(lines), "| characters:", sum(len(l) for l in lines))
print("Headings:", [l for l in lines if l.startswith("## ")][:15])
print("\n".join(lines[:25]))

In [ ]:
files = sorted(f for f in RAW_DIR.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW_DIR), "bytes:", f.stat().st_size)
assert files, "No files found"

In [ ]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
# Run ONCE after Cell 1 and Cell 3
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

# Select the exact source before parsing.
# Never automatically take the first file by extension.
def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(
                f"Source not found: {selected}"
            )
        return selected

    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print(
        "Candidates:",
        [str(p.relative_to(RAW_DIR)) for p in matches]
    )
    if len(matches) != 1:
        raise ValueError(
            "Select exactly one source. "
            "Set relative_path to an exact file under RAW_DIR."
        )
    return matches[0]

def source_hash(path):
    return hashlib.sha256(
        Path(path).read_bytes()
    ).hexdigest()

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )
    print("Saved lineage:", target)
    return manifest

# Confirmation messages
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")

In [ ]:
# CELL 4A — COMMON LINEAGE REGISTRY
# REQUIRED: Run once before selected Part B pathways
from pathlib import Path
lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    # Validate source
    if not source.is_file():
        raise FileNotFoundError(
            f"Source file not found: {source}"
        )
    # Validate prepared output
    if not output.is_file():
        raise FileNotFoundError(
            f"Prepared output not found: {output}"
        )
    if output.stat().st_size == 0:
        raise ValueError(
            f"Prepared output is empty: {output}"
        )
    # Validate processing description
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError(
            "A processing description is required."
        )
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError(
            "Replace the placeholder with the actual method."
        )
    # Ensure source and output belong to project workspace
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    # Same source/output pair is updated, not duplicated
    key = (str(source), str(output))
    lineage_entries[key] = {
        "source": source,
        "output": output,
        "processing": processing
    }
    print(
        "Registered:",
        source.name,
        "->",
        output.name
    )

print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

In [ ]:
!pip -q install pypdf
from pypdf import PdfReader
import re

f = select_source(relative_path="0c-esp-wroom-02_datasheet_en.pdf")
reader = PdfReader(str(f))
pages = [(i + 1, p.extract_text() or "") for i, p in enumerate(reader.pages)]
text = "\n".join(t for _, t in pages)
clean = re.sub(r"[ \t]+", " ", text).strip()

print("Source:", f.name)
print("Pages:", len(pages), "| chars:", len(clean))
print("Empty pages:", [p for p, t in pages if not t.strip()])
print(repr(clean[:300]))

In [ ]:
import re

def fixed_chunks(s, size=500):
    return [s[i:i+size] for i in range(0, len(s), size)]

def paragraph_chunks(s, size=500):
    parts, current = [], ""
    for seg in re.split(r"\n\s*\n", s):
        seg = seg.strip()
        if not seg:
            continue
        if len(seg) > size:
            if current:
                parts.append(current); current = ""
            parts.extend(fixed_chunks(seg, size))
        elif len(current) + len(seg) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + seg
        else:
            if current:
                parts.append(current)
            current = seg
    if current:
        parts.append(current)
    return parts

def page_aware_chunks(pages, size=500):
    """pages = [(page_no, text)] -> [(page_no, chunk)]; never crosses a page."""
    out = []
    for page_no, content in pages:
        text = re.sub(r"[ \t]+", " ", content).strip()
        if text:
            out.extend((page_no, c) for c in paragraph_chunks(text, size))
    return out

print("Helpers ready.")

In [ ]:
fixed = fixed_chunks(clean, 500)
para = paragraph_chunks(clean, 500)
paged = page_aware_chunks(pages, 500)

def avg_len(chunks):
    return round(sum(len(c) for c in chunks) / len(chunks))

print("Source:", f.name)
print(f"Fixed          : {len(fixed):5d} chunks | avg {avg_len(fixed)} chars | with page number: 0%")
print(f"Paragraph-aware: {len(para):5d} chunks | avg {avg_len(para)} chars | with page number: 0%")
print(f"Page-aware     : {len(paged):5d} chunks | avg {avg_len([c for _, c in paged])} chars | with page number: 100%")
print("Empty pages (no text):", [p for p, t in pages if not t.strip()])

print("\nFixed sample:\n", repr(fixed[5][:200]))
print("\nPage-aware sample:\n", paged[5])

In [ ]:
LIGATURES = {"ﬁ": "fi", "ﬂ": "fl", "ﬀ": "ff", "ﬃ": "ffi", "ﬄ": "ffl"}

def clean_page(s):
    for k, v in LIGATURES.items():
        s = s.replace(k, v)
    s = s.replace("T able", "Table")
    s = re.sub(r"\bT (?=[a-z]{2,}\b)", "T", s)
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r" ?\n ?", "\n", s)
    return s.strip()

def line_packed_chunks(pages, size=500):
    """Page-aware; packs whole lines up to `size`; hard-cuts only at a space."""
    out = []
    for page_no, content in pages:
        current = ""
        for line in clean_page(content).split("\n"):
            line = line.strip()
            if not line:
                continue
            while len(line) > size:
                cut = line.rfind(" ", 0, size)
                cut = cut if cut > 0 else size
                if current:
                    out.append((page_no, current)); current = ""
                out.append((page_no, line[:cut].strip()))
                line = line[cut:].strip()
            if not line:
                continue
            if len(current) + len(line) + (1 if current else 0) <= size:
                current += ("\n" if current else "") + line
            else:
                out.append((page_no, current))
                current = line
        if current:
            out.append((page_no, current))
    return out

packed = line_packed_chunks(pages, 500)

def text_of(c):
    return c[1] if isinstance(c, tuple) else c

def starts_lower(chunks):  # crude proxy for "begins mid-sentence/mid-word"
    return sum(1 for c in chunks if text_of(c)[:1].islower())

print(f"{'Method':18s}{'chunks':>8s}{'avg len':>9s}{'start lowercase':>17s}{'page no.':>10s}")
for name, ch, pg in [("fixed", fixed, "0%"), ("paragraph-aware", para, "0%"),
                     ("page-aware (old)", paged, "100%"), ("page + line-packed", packed, "100%")]:
    avg = round(sum(len(text_of(c)) for c in ch) / len(ch))
    print(f"{name:18s}{len(ch):8d}{avg:9d}{starts_lower(ch):17d}{pg:>10s}")
print("\nSample:", packed[10])

In [ ]:
for key in ["absolute maximum", "electrical characteristics", "pin definitions", "pin layout"]:
    hits = [p for p, t in pages if key in clean_page(t).lower()]
    print(f"{key:28s} -> pages {hits}")

In [ ]:
page_map = dict(pages)
print(clean_page(page_map[23]))

In [ ]:
import json
from pypdf import PdfReader

CHUNK_SIZE = 500
APPROACH = "page/section-aware-line-packed-char-500; ligature-clean; split-T-repair"
SOURCES_B = [
    "0c-esp-wroom-02_datasheet_en.pdf",
    "esp8266-technical_reference_en.pdf",
    "esp8266_rtos_sdk_get_started.txt",
]
DOC_VERSION = {
    "0c-esp-wroom-02_datasheet_en.pdf": "Version 3.7 (2025.11)",
    "esp8266-technical_reference_en.pdf": "Version 1.7",
    "esp8266_rtos_sdk_get_started.txt": "docs.espressif.com 'latest' (see retrieval date in file header)",
}

def read_units(src):
    """PDF -> [(page_no, text)]; TXT -> [(section_title, text)] split on '## ' lines."""
    if src.suffix.lower() == ".pdf":
        return [(i + 1, p.extract_text() or "") for i, p in enumerate(PdfReader(str(src)).pages)]
    if src.suffix.lower() == ".txt":
        units, title, buf = [], "(start)", []
        for line in src.read_text(encoding="utf-8").splitlines():
            if line.startswith("## "):
                if buf:
                    units.append((title, "\n".join(buf)))
                title, buf = line[3:].strip(), []
            else:
                buf.append(line)
        if buf:
            units.append((title, "\n".join(buf)))
        return units
    raise ValueError("Expected a PDF or TXT source")

for name in SOURCES_B:
    src = select_source(relative_path=name)
    is_pdf = src.suffix.lower() == ".pdf"
    units = read_units(src)
    print(f"\n=== {name} | {'pages' if is_pdf else 'sections'}: {len(units)} ===")
    print("First unit:", repr(clean_page(units[0][1])[:300]))

    chunks = line_packed_chunks(units, CHUNK_SIZE)
    digest = source_hash(src)
    prepared_path = PROCESSED_DIR / f"prepared_text_{src.stem}.jsonl"

    with prepared_path.open("w", encoding="utf-8") as out:
        for i, (unit, text) in enumerate(chunks, 1):
            out.write(json.dumps({
                "record_id": f"{src.stem}-{i:04d}",
                "source": str(src.relative_to(RAW_DIR)),
                "source_sha256": digest,
                "document_version": DOC_VERSION.get(name),
                "page": unit if is_pdf else None,
                "section": None if is_pdf else unit,
                "text": text,
                "processing": APPROACH,
            }, ensure_ascii=False) + "\n")

    # B3 basic output contract
    assert prepared_path.exists(), "Prepared output missing."
    assert prepared_path.stat().st_size > 0, "Prepared output is empty."
    with prepared_path.open(encoding="utf-8") as fh:
        rows = [json.loads(l) for l in fh if l.strip()]
    assert len(rows) == len(chunks), "Exported count mismatch"
    assert all(r["text"].strip() for r in rows), "Empty chunk found"
    if is_pdf:
        assert all(isinstance(r["page"], int) and 1 <= r["page"] <= len(units) for r in rows), "Bad page number"
    else:
        assert all(isinstance(r["section"], str) and r["section"] for r in rows), "Missing section label"

    print("Chunks exported:", len(rows), "| size:", prepared_path.stat().st_size, "bytes")
    print("Units covered:", len({r['page'] if is_pdf else r['section'] for r in rows}), "of", len(units))
    print("Sample:", rows[len(rows) // 2])

    # B4.1 lineage registration (Pathway A)
    register_lineage(source=src, output=prepared_path, processing=APPROACH)

print("\nRegistered entries:", len(lineage_entries))

# Cell 6A and 6B

In [ ]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST
# REQUIRED: Run after all selected pathways are registered
assert lineage_entries, (
 "No lineage entries registered. "
 "Register at least one selected pathway first."
)
lineage_records = []
for i, entry in enumerate(lineage_entries.values(), 1):
    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]
    # Validate registered files again
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"
    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(
            source.relative_to(RAW_DIR.resolve())
        ),
        "source_sha256": source_hash(source),
        "prepared_output": str(
            output.relative_to(BASE_DIR.resolve())
        ),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Not verified",
        "access_rights": "Not verified",
        "status": "Prepared; quality review pending"
    })
# Reuse the common manifest writer from Cell 4
manifest = save_manifest(lineage_records)
print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print(
 "Saved to:",
 BASE_DIR / "data" / "week3_manifest.json"
)

In [ ]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST
# REQUIRED: Run after Cell 6A
import json
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), (
 "Common lineage manifest not found."
)
with manifest_path.open("r", encoding="utf-8") as file:
 saved_manifest = json.load(file)
saved_records = saved_manifest.get("records", [])
assert saved_records, (
 "No lineage records found in saved manifest."
)
assert len(saved_records) == len(lineage_entries), (
 "Saved lineage count does not match registration."
)
for record in saved_records:
 source = RAW_DIR / record["source_filename"]
 output = BASE_DIR / record["prepared_output"]
 assert source.is_file(), f"Missing source: {source}"
 assert output.is_file(), f"Missing output: {output}"
 assert source_hash(source) == record["source_sha256"], (
 f"Source checksum mismatch: {source}"
 )
 assert source_hash(output) == record["prepared_sha256"], (
 f"Prepared output checksum mismatch: {output}"
 )
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

In [ ]:
!cd /content/week3_project && zip -r ../part_b_outputs.zip data/processed data/week3_manifest.json

In [ ]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
# Run ONCE after completing Part B
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
# Step 1: Define standard project paths
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)
# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}
# Step 2: Calculate SHA-256 checksum
def sha256_file(path):
 return hashlib.sha256(
 Path(path).read_bytes()
 ).hexdigest()
# Step 3: Resolve an explicitly selected source
def resolve_source(relative_path, folder=False):
 source = RAW_DIR / relative_path
 if folder:
     if not source.is_dir():
         raise FileNotFoundError(
             f"Source folder not found: {source}"
         )
 else:
     if not source.is_file():
         raise FileNotFoundError(
             f"Source file not found: {source}"
         )
 return source
# Step 4: Execute the selected V1 data pipeline
def execute_pipeline(
 kind,
 source,
 inspect_fn,
 prepare_fn,
 configuration
):
 source = Path(source)
 # INGEST: Identify relevant source files
 if kind == "images":
     inputs = sorted(
         p for p in source.iterdir()
         if p.is_file()
         and p.suffix.lower() in {
             ".png", ".jpg", ".jpeg", ".webp"
         }
     )
 else:
     inputs = [source]
 if not inputs or any(not p.is_file() for p in inputs):
     raise ValueError(
         "No valid source files to ingest."
     )
 # INSPECT: Apply data-type-specific inspection
 inspection = inspect_fn(inputs, configuration)
 # Define output directory
 output_dir = PIPE_DIR / kind
 output_dir.mkdir(parents=True, exist_ok=True)
 # PREPARE / STORE: Apply selected transformation
 outputs, mappings, method = prepare_fn(
     inputs,
     output_dir,
     configuration
 )
 outputs = [Path(p) for p in outputs]
 # VALIDATE: Check prepared outputs
 if not outputs:
     raise ValueError(
         "No prepared outputs were produced."
     )
 for output in outputs:
     if not output.is_file():
         raise FileNotFoundError(
             f"Prepared output missing: {output}"
         )
     if output.stat().st_size == 0:
         raise ValueError(
             f"Prepared output is empty: {output}"
         )
 # Record basic validation evidence
 validation = {
     "output_count": len(outputs),
     "output_sha256": [
         sha256_file(p) for p in outputs
     ],
     "output_bytes": [
         p.stat().st_size for p in outputs
     ]
 }
 # RECORD LINEAGE: Create pipeline execution record
 record = {
     "pipeline_version": "week3-v1",
     "generated_at_utc": datetime.now(
         timezone.utc
     ).isoformat(),
     "data_type": kind,
     "configuration": configuration,
     "processing_method": method,
     "sources": [
         {
             "path": str(p),
             "sha256": sha256_file(p)
         }
         for p in inputs
     ],
     "outputs": [
         str(p) for p in outputs
     ],
     "source_output_mapping": mappings,
     "inspection": inspection,
     "validation": validation,
     "source_version": "Not verified",
     "access_rights": (
         "Not verified — check before sharing"
     )
 }
 # Save pipeline manifest
 manifest_path = output_dir / "pipeline_manifest.json"
 manifest_path.write_text(
     json.dumps(
         record,
         indent=2,
         ensure_ascii=False
     ),
     encoding="utf-8"
 )
 print(
     f"SUCCESS: {kind} | "
     f"Inputs: {len(inputs)} | "
     f"Outputs: {len(outputs)}"
 )
 print("Manifest:", manifest_path)
 return record
# Step 5: Register and run a selected pathway
def register_and_run(
 kind,
 source,
 inspect_fn,
 prepare_fn,
 configuration
):
 # Keep the runner for repeatability verification
 def runner():
     return execute_pipeline(
         kind,
         source,
         inspect_fn,
         prepare_fn,
         configuration
     )
 first = runner()
 PIPELINE_RUNS[kind] = {
     "first": first,
     "runner": runner
 }
 return first
print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")

In [ ]:
!pip -q install pypdf
import re, json
from pypdf import PdfReader

CONFIG_A = {
    "chunk_size": 500,
    "method": "page/section-aware-line-packed",
    "cleaning": "ligatures; split-T repair; whitespace",
}

# kind -> (relative path under data/raw, document version for FR5)
DOCS = {
    "documents_datasheet": ("0c-esp-wroom-02_datasheet_en.pdf", "Version 3.7 (2025.11)"),
    "documents_trm": ("esp8266-technical_reference_en.pdf", "Version 1.7"),
    "documents_sdk": ("esp8266_rtos_sdk_get_started.txt",
                      "docs.espressif.com 'latest' (see retrieval date in file header)"),
}

LIGATURES = {"ﬁ": "fi", "ﬂ": "fl", "ﬀ": "ff", "ﬃ": "ffi", "ﬄ": "ffl"}

def clean_page(s):
    for k, v in LIGATURES.items():
        s = s.replace(k, v)
    s = s.replace("T able", "Table")
    s = re.sub(r"\bT (?=[a-z]{2,}\b)", "T", s)
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r" ?\n ?", "\n", s)
    return s.strip()

def line_packed_chunks(units, size=500):
    out = []
    for unit, content in units:
        current = ""
        for line in clean_page(content).split("\n"):
            line = line.strip()
            if not line:
                continue
            while len(line) > size:
                cut = line.rfind(" ", 0, size)
                cut = cut if cut > 0 else size
                if current:
                    out.append((unit, current)); current = ""
                out.append((unit, line[:cut].strip()))
                line = line[cut:].strip()
            if not line:
                continue
            if len(current) + len(line) + (1 if current else 0) <= size:
                current += ("\n" if current else "") + line
            else:
                out.append((unit, current))
                current = line
        if current:
            out.append((unit, current))
    return out

_UNIT_CACHE = {}   # PDF extraction is slow; Cell 9 reruns the pipeline

def read_units(src):
    key = str(src)
    if key not in _UNIT_CACHE:
        if src.suffix.lower() == ".pdf":
            _UNIT_CACHE[key] = [(i + 1, p.extract_text() or "")
                                for i, p in enumerate(PdfReader(key).pages)]
        elif src.suffix.lower() == ".txt":
            units, title, buf = [], "(start)", []
            for line in src.read_text(encoding="utf-8").splitlines():
                if line.startswith("## "):
                    if buf:
                        units.append((title, "\n".join(buf)))
                    title, buf = line[3:].strip(), []
                else:
                    buf.append(line)
            if buf:
                units.append((title, "\n".join(buf)))
            _UNIT_CACHE[key] = units
        else:
            raise ValueError("Expected a PDF or TXT source")
    return _UNIT_CACHE[key]

def inspect_documents(inputs, cfg):
    src = inputs[0]
    units = read_units(src)
    chars = sum(len(t) for _, t in units)
    if chars == 0:
        raise ValueError("No extractable text; scanned PDFs may need OCR.")
    return {"source": src.name,
            "unit_type": "page" if src.suffix.lower() == ".pdf" else "section",
            "units": len(units), "characters": chars,
            "empty_units": [u for u, t in units if not t.strip()]}

def prepare_documents(inputs, out, cfg):
    src = inputs[0]
    is_pdf = src.suffix.lower() == ".pdf"
    units = read_units(src)
    chunks = line_packed_chunks(units, int(cfg["chunk_size"]))
    if not chunks:
        raise ValueError("No text chunks produced")
    digest = sha256_file(src)
    target = out / "prepared_text.jsonl"
    with target.open("w", encoding="utf-8") as h:
        for i, (unit, text) in enumerate(chunks, 1):
            h.write(json.dumps({
                "record_id": f"{src.stem}-{i:04d}",
                "source": str(src.relative_to(RAW_DIR)),
                "source_sha256": digest,
                "document_version": cfg.get("document_version", "Not verified"),
                "page": unit if is_pdf else None,
                "section": None if is_pdf else unit,
                "text": text,
            }, ensure_ascii=False) + "\n")
    # read-back check (stage contract: Validate)
    with target.open(encoding="utf-8") as h:
        rows = [json.loads(l) for l in h if l.strip()]
    if len(rows) != len(chunks) or not all(r["text"].strip() for r in rows):
        raise ValueError("Exported chunks do not match prepared chunks")
    if is_pdf and not all(isinstance(r["page"], int) for r in rows):
        raise ValueError("A chunk is missing its page number")
    if not is_pdf and not all(r["section"] for r in rows):
        raise ValueError("A chunk is missing its section label")
    mapping = [{"source": str(src), "output": str(target)}]
    return [target], mapping, f"{cfg['method']} chunks of {cfg['chunk_size']} chars; {cfg['cleaning']}"

for kind, (rel, version) in DOCS.items():
    register_and_run(kind, resolve_source(rel), inspect_documents, prepare_documents,
                     {**CONFIG_A, "document_version": version})

print("Cell 8A completed successfully.")

In [ ]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification
# Run after your selected Cell 8A–8F pathway(s).
# Step 1: Ensure at least one pipeline has been registered
assert PIPELINE_RUNS, (
 "No pipeline runs registered. "
 "Run at least one selected Cell 8 pathway first."
)
print("--- Repeatability Verification ---")
# Step 2: Rerun each selected pipeline
for kind, entry in PIPELINE_RUNS.items():
 first = entry["first"]
 # Execute the same pipeline again
 second = entry["runner"]()
 # Step 3: Verify input provenance
 assert first["sources"] == second["sources"], (
 f"Source fingerprint changed for {kind}"
 )
 # Step 4: Verify prepared output fingerprints
 assert first["validation"] == second["validation"], (
 f"Prepared output changed for {kind}"
 )
 # Step 5: Verify source-to-output mapping
 assert (
 first["source_output_mapping"]
 == second["source_output_mapping"]
 ), (
 f"Source-to-output mapping changed for {kind}"
 )
 print(
 f"PASS: {kind} — source fingerprints, "
 "prepared outputs and mappings match"
 )
# Step 6: Display final verification message
print("\nAll selected pipelines passed repeatability smoke checks.")

In [ ]:
for kind in DOCS:
    p = PIPE_DIR / kind
    print(kind, [x.name for x in p.iterdir()])
    first = json.loads((p / "prepared_text.jsonl").read_text(encoding="utf-8").splitlines()[3])
    print(first, "\n")

In [ ]:
!cd /content/week3_project && zip -r ../part_c_outputs.zip data/pipeline_v1

In [ ]:
print(clean_page(page_map[8]))
print("=" * 60)
print(clean_page(page_map[14]))

In [ ]:
for key in ["maximum", "ratings", "storage temperature", "operating temperature", "3.6 v", "3.3 v"]:
    print(f"{key:24s} -> pages {[p for p, t in pages if key in clean_page(t).lower()]}")

In [ ]:
print(clean_page(page_map[7]))
print("=" * 60)
print(clean_page(page_map[13]))